# Predictive Modeling — Payment Classification
**Target:** `Payment` (Not Paid vs Paid)  
**Approach:** ML (XGBoost + RandomForest) + DL (Neural Network)  
**Imbalance Fix:** SMOTE

## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Imbalance
from imblearn.over_sampling import SMOTE

# ML Models
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Metrics
from sklearn.metrics import (
    classification_report, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# DL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('All imports done!')
print('TensorFlow version:', tf.__version__)

## Step 2 — Load Data

In [ ]:
df = pd.read_excel('Predictive_modeling.xlsx')
print('Shape:', df.shape)
df.head(3)

## Step 3 — Drop Unimportant Columns

**Drop reasons:**
- `ID` → unique identifier, no predictive value
- `City`, `device_model`, `platform` → too many cardinality / noise
- `Nature_Sign` → low info
- `*_Bucket`, `City_Group` → derived from numeric cols (redundant)
- Granular sub-columns (RE_Residential, Equity_MF, Debt_EPF etc.) → already summed in parent cols (Real_Estate, Equity, Debt etc.)

In [ ]:
drop_cols = [
    # Identifier & noise
    'ID', 'City', 'device_model', 'platform', 'Nature_Sign', 'non_Other',
    # Derived bucket cols (redundant)
    'Income_Bucket', 'Age_Bucket', 'FBS_Bucket', 'Cibil_Bucket', 'City_Group',
    # Granular sub-items (summed in parent cols)
    'RE_Residential', 'RE_Commercial',
    'Equity_MF', 'Equity_ESOPs', 'Equity_Stocks',
    'Debt_EPF', 'Debt_FD', 'Debt_GovtSchemes', 'Debt_HybridMF',
    'Debt_MF', 'Debt_NPS', 'Debt_Other', 'Debt_PPF', 'Debt_Saving',
    'Gold_Digital', 'Gold_Physical', 'Gold_SGB',
    'AltInv_Crypto', 'AltInv_P2P_OR_LoansGiven', 'AltInv_Silver',
    'Housing_Loan', 'Auto_Loan', 'Personal_Loan', 'Consumer_Loan',
    'Credit_Card', 'Education_Loan', 'Gold_Loan', 'Other_loan'
]

df.drop(columns=drop_cols, inplace=True)
print('Shape after drop:', df.shape)
print('Remaining columns:', df.columns.tolist())

## Step 4 — Target Variable
Binary classification: **0 = Not Paid**, **1 = Paid (any)**

In [ ]:
df['target'] = (df['Payment'] != 'Not Paid').astype(int)
df.drop(columns=['Payment'], inplace=True)

print('Target distribution:')
print(df['target'].value_counts())
print()
print('Class %:')
print(df['target'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize imbalance
plt.figure(figsize=(5, 3))
df['target'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Not Paid (0)', 'Paid (1)'], rotation=0)
plt.title('Class Imbalance (Before SMOTE)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Step 5 — Preprocessing
1. Categorical → Label Encode
2. Numeric nulls → Median Impute

In [ ]:
# 5a. Encode categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')
    df[col] = le.fit_transform(df[col].astype(str))

# 5b. Impute numeric nulls with median
num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'target']
imp = SimpleImputer(strategy='median')
df[num_cols] = imp.fit_transform(df[num_cols])

print('\nNull count after impute:', df.isnull().sum().sum())
print('Final shape:', df.shape)

## Step 6 — Train/Test Split + SMOTE (Imbalance Fix)

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# Split first, then SMOTE only on train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Train target dist (before SMOTE):', y_train.value_counts().to_dict())

# SMOTE — oversample minority class
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nTrain target dist (after SMOTE) :', pd.Series(y_train_res).value_counts().to_dict())

In [ ]:
# Scale for DL model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

## Step 7 — ML Model 1: XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test)

print('='*50)
print('XGBoost Results')
print('='*50)
print(classification_report(y_test, y_pred_xgb, target_names=['Not Paid', 'Paid']))
print('Weighted F1:', round(f1_score(y_test, y_pred_xgb, average='weighted'), 4))

In [ ]:
# XGBoost Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Paid', 'Paid'])
disp.plot(cmap='Blues')
plt.title('XGBoost — Confusion Matrix')
plt.show()

In [ ]:
# XGBoost Feature Importance
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(7, 5), color='steelblue')
plt.title('XGBoost — Feature Importance')
plt.tight_layout()
plt.show()

## Step 8 — ML Model 2: Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_res, y_train_res)
y_pred_rf = rf_model.predict(X_test)

print('='*50)
print('Random Forest Results')
print('='*50)
print(classification_report(y_test, y_pred_rf, target_names=['Not Paid', 'Paid']))
print('Weighted F1:', round(f1_score(y_test, y_pred_rf, average='weighted'), 4))

In [ ]:
# RF Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Not Paid', 'Paid'])
disp_rf.plot(cmap='Greens')
plt.title('Random Forest — Confusion Matrix')
plt.show()

## Step 9 — DL Model: Neural Network (Keras)

In [ ]:
# Class weights for DL (alternate imbalance handling)
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_res)
weights = compute_class_weight('balanced', classes=classes, y=y_train_res)
class_weight_dict = dict(zip(classes, weights))
print('Class weights:', class_weight_dict)

input_dim = X_train_scaled.shape[1]
print('Input features:', input_dim)

In [ ]:
# Build Neural Network
tf.random.set_seed(42)

dl_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')  # binary output
])

dl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

dl_model.summary()

In [ ]:
# Train Neural Network
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = dl_model.fit(
    X_train_scaled, y_train_res,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Loss Curve')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Accuracy Curve')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# DL Predictions
y_pred_dl_prob = dl_model.predict(X_test_scaled).flatten()
y_pred_dl = (y_pred_dl_prob >= 0.5).astype(int)

print('='*50)
print('Neural Network Results')
print('='*50)
print(classification_report(y_test, y_pred_dl, target_names=['Not Paid', 'Paid']))
print('Weighted F1:', round(f1_score(y_test, y_pred_dl, average='weighted'), 4))

In [ ]:
# DL Confusion Matrix
cm_dl = confusion_matrix(y_test, y_pred_dl)
disp_dl = ConfusionMatrixDisplay(confusion_matrix=cm_dl, display_labels=['Not Paid', 'Paid'])
disp_dl.plot(cmap='Oranges')
plt.title('Neural Network — Confusion Matrix')
plt.show()

## Step 10 — Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest', 'Neural Network'],
    'Weighted F1': [
        round(f1_score(y_test, y_pred_xgb, average='weighted'), 4),
        round(f1_score(y_test, y_pred_rf,  average='weighted'), 4),
        round(f1_score(y_test, y_pred_dl,  average='weighted'), 4)
    ],
    'Minority F1 (Paid)': [
        round(f1_score(y_test, y_pred_xgb, average=None)[1], 4),
        round(f1_score(y_test, y_pred_rf,  average=None)[1], 4),
        round(f1_score(y_test, y_pred_dl,  average=None)[1], 4)
    ]
})

print(results.to_string(index=False))

# Plot
results.set_index('Model')[['Weighted F1', 'Minority F1 (Paid)']].plot(
    kind='bar', figsize=(8, 4), color=['steelblue', 'tomato']
)
plt.title('Model Comparison — F1 Scores')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()